# NSE Canara Bank — Quarterly Filing Downloader

Download **standalone quarterly** filings from NSE for **01 Jan 2014 – 08 Nov 2026**. Original XBRL XML is preserved when available; older HTML filings are downloaded and tables extracted to JSON, CSV and **non-XBRL** XML. Revised submissions are retained.

> Requires internet access and may require NSE browser verification. Run locally on your MacBook. NSE may restrict automated requests.

## 1. Install dependencies
Run once. Restart the notebook kernel if prompted.

In [1]:
%pip install -q playwright
import subprocess, sys
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
|                                                                                |   0% of 185.8 MiB
|■■■■■■■■                                                                        |  10% of 185.8 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 185.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 185.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 185.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 185.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 185.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 185.8 MiB
|■■■■■■■■■■■■■■

CompletedProcess(args=['/Users/rahul/finetune/finetune_llm/venv/bin/python', '-m', 'playwright', 'install', 'chromium'], returncode=0)

## 2. Downloader implementation
This cell is self-contained; no separate `.py` file is required.

In [2]:
#!/usr/bin/env python3
"""
Download Canara Bank (CANBK) standalone quarterly financial-result filings
from NSE for calendar filing dates 2014-01-01 through 2026-12-31.

Retrieval priority:
  1. Original NSE XBRL XML, when the API supplies a real .xml URL.
  2. Original detailed HTML, when no XBRL URL exists.
  3. A clearly labelled reconstructed table XML + JSON/CSV extracted from
     legacy HTML. This is NOT official XBRL and does not invent taxonomy facts.

All submissions are retained, including revised filings. The manifest records
each submission separately. The script uses a visible Playwright Chromium
session to establish NSE cookies, then calls the observed NSE JSON endpoint.
"""

from __future__ import annotations

import argparse
import asyncio
import csv
import hashlib
import json
import logging
import re
import sys
import time
import xml.etree.ElementTree as ET
from datetime import datetime, date
from html.parser import HTMLParser
from pathlib import Path
from typing import Any
from urllib.parse import urlencode, urlparse

from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


BASE = "https://www.nseindia.com"
RESULTS_PAGE = (
    "https://www.nseindia.com/companies-listing/"
    "corporate-filings-financial-results"
)
API_PATH = "/api/corporates-financial-results"

SYMBOL = "CANBK"
ISSUER = "Canara Bank"
ISIN = "INE476A01014"

# Calendar filing-date window, inclusive.
FROM_DATE = "01-01-2014"
TO_DATE = "08-11-2026"

OUT = Path("canara_bank_xbrl")
REQUEST_DELAY_SECONDS = 0.6
MAX_RETRIES = 3

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Referer": RESULTS_PAGE,
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    ),
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("nse-downloader")


def parse_nse_date(value: Any) -> datetime | None:
    if not value or value == "-":
        return None
    value = str(value).strip()
    for fmt in (
        "%d-%b-%Y %H:%M:%S",
        "%d-%b-%Y %H:%M",
        "%d-%b-%Y",
        "%d-%m-%Y %H:%M:%S",
        "%d-%m-%Y %H:%M",
        "%d-%m-%Y",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d",
    ):
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            pass
    return None


def filing_datetime(record: dict[str, Any]) -> datetime | None:
    # NSE's date filter is a filing-date filter. Prefer filingDate, then
    # broadcastDate, and finally exchange dissemination time.
    for key in ("filingDate", "broadCastDate", "exchdisstime"):
        parsed = parse_nse_date(record.get(key))
        if parsed:
            return parsed
    return None


def safe_component(value: Any) -> str:
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(value or "unknown"))
    return text.strip("._-") or "unknown"


def filing_identity(record: dict[str, Any]) -> str:
    # Do not deduplicate by quarter: every submission/revision is retained.
    stable = {
        "seqNumber": record.get("seqNumber"),
        "filingDate": record.get("filingDate"),
        "broadCastDate": record.get("broadCastDate"),
        "xbrl": record.get("xbrl"),
        "resultDetailedDataLink": record.get("resultDetailedDataLink"),
        "params": record.get("params"),
    }
    raw = json.dumps(stable, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:12]


def filing_folder(record: dict[str, Any]) -> Path:
    dt = filing_datetime(record)
    year = dt.year if dt else "unknown_year"
    folder = OUT / str(year)
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def filename_stem(record: dict[str, Any]) -> str:
    period_end = parse_nse_date(record.get("toDate"))
    period = period_end.strftime("%Y%m%d") if period_end else "period_unknown"
    broadcast = parse_nse_date(record.get("broadCastDate"))
    broadcast_text = broadcast.strftime("%Y%m%d_%H%M%S") if broadcast else "broadcast_unknown"
    seq = safe_component(record.get("seqNumber"))
    quarter = safe_component(record.get("relatingTo"))
    return f"{SYMBOL}_{period}_{quarter}_{broadcast_text}_seq-{seq}_{filing_identity(record)}"


def valid_nse_archive_url(url: Any, suffix: str) -> bool:
    if not isinstance(url, str) or not url.startswith("https://"):
        return False
    parsed = urlparse(url)
    return (
        parsed.hostname == "nsearchives.nseindia.com"
        and parsed.path.lower().endswith(suffix)
        and not parsed.path.endswith("/-")
    )


def extract_records(payload: Any) -> list[dict[str, Any]]:
    """Handle either a top-level list or common NSE wrapper objects."""
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]

    if isinstance(payload, dict):
        for key in ("data", "results", "records", "financialResults"):
            value = payload.get(key)
            if isinstance(value, list):
                return [x for x in value if isinstance(x, dict)]
        # Some NSE endpoints wrap the list one level deeper.
        for value in payload.values():
            if isinstance(value, list) and value and isinstance(value[0], dict):
                if "symbol" in value[0] or "seqNumber" in value[0]:
                    return value

    raise ValueError(
        "Unexpected NSE API JSON shape. Raw response was saved for inspection."
    )


def filter_records(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    start = datetime.strptime(FROM_DATE, "%d-%m-%Y").date()
    end = datetime.strptime(TO_DATE, "%d-%m-%Y").date()
    kept: list[dict[str, Any]] = []
    seen: set[str] = set()

    for row in records:
        if row.get("symbol") != SYMBOL:
            continue
        # In this NSE API, Non-Consolidated means standalone.
        if row.get("consolidated") != "Non-Consolidated":
            continue
        if row.get("period") and row.get("period") != "Quarterly":
            continue

        dt = filing_datetime(row)
        if not dt or not (start <= dt.date() <= end):
            continue

        identity = filing_identity(row)
        if identity in seen:
            continue
        seen.add(identity)
        kept.append(row)

    return sorted(
        kept,
        key=lambda r: (
            filing_datetime(r) or datetime.min,
            str(r.get("seqNumber") or ""),
        ),
    )


class TableExtractor(HTMLParser):
    """Small dependency-free HTML table extractor."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self.tables: list[list[list[str]]] = []
        self._table: list[list[str]] | None = None
        self._row: list[str] | None = None
        self._cell: list[str] | None = None
        self._in_cell = False

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        tag = tag.lower()
        if tag == "table":
            self._table = []
        elif tag == "tr" and self._table is not None:
            self._row = []
        elif tag in ("td", "th") and self._row is not None:
            self._cell = []
            self._in_cell = True
        elif tag == "br" and self._in_cell and self._cell is not None:
            self._cell.append(" ")

    def handle_endtag(self, tag: str) -> None:
        tag = tag.lower()
        if tag in ("td", "th") and self._in_cell:
            text = re.sub(r"\s+", " ", "".join(self._cell or [])).strip()
            if self._row is not None:
                self._row.append(text)
            self._cell = None
            self._in_cell = False
        elif tag == "tr" and self._row is not None:
            if any(cell.strip() for cell in self._row):
                if self._table is not None:
                    self._table.append(self._row)
            self._row = None
        elif tag == "table" and self._table is not None:
            if self._table:
                self.tables.append(self._table)
            self._table = None

    def handle_data(self, data: str) -> None:
        if self._in_cell and self._cell is not None:
            self._cell.append(data)


def save_reconstructed_tables(html_bytes: bytes, record: dict[str, Any], folder: Path) -> dict[str, str]:
    """
    Extract HTML tables and save a neutral table representation.

    The XML produced here is intentionally NOT tagged as XBRL. NSE's legacy
    HTML pages do not supply enough taxonomy/context metadata to recreate
    an official XBRL instance safely.
    """
    text = html_bytes.decode("utf-8", errors="replace")
    parser = TableExtractor()
    parser.feed(text)

    stem = filename_stem(record)
    json_path = folder / f"{stem}_tables.json"
    csv_path = folder / f"{stem}_tables.csv"
    xml_path = folder / f"{stem}_reconstructed_tables.xml"

    table_data = {
        "document_type": "reconstructed_html_table_extraction",
        "is_official_xbrl": False,
        "source_url": record.get("resultDetailedDataLink"),
        "filing_metadata": record,
        "tables": [
            {
                "table_number": i + 1,
                "rows": rows,
            }
            for i, rows in enumerate(parser.tables)
        ],
    }
    json_path.write_text(
        json.dumps(table_data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["table_number", "row_number", "cell_number", "value"])
        for table_no, table in enumerate(parser.tables, start=1):
            for row_no, row in enumerate(table, start=1):
                for cell_no, cell in enumerate(row, start=1):
                    writer.writerow([table_no, row_no, cell_no, cell])

    root = ET.Element(
        "ReconstructedFinancialTables",
        attrib={
            "documentType": "HTML_TABLE_EXTRACTION",
            "isOfficialXBRL": "false",
            "symbol": SYMBOL,
        },
    )
    source = ET.SubElement(root, "SourceURL")
    source.text = str(record.get("resultDetailedDataLink") or "")
    metadata = ET.SubElement(root, "FilingMetadata")
    for key, value in record.items():
        node = ET.SubElement(metadata, safe_component(key))
        node.text = "" if value is None else str(value)

    tables_node = ET.SubElement(root, "Tables")
    for table_no, rows in enumerate(parser.tables, start=1):
        table_node = ET.SubElement(tables_node, "Table", number=str(table_no))
        for row_no, row in enumerate(rows, start=1):
            row_node = ET.SubElement(table_node, "Row", number=str(row_no))
            for cell_no, cell in enumerate(row, start=1):
                cell_node = ET.SubElement(row_node, "Cell", number=str(cell_no))
                cell_node.text = cell

    ET.indent(root, space="  ")
    ET.ElementTree(root).write(
        xml_path,
        encoding="utf-8",
        xml_declaration=True,
    )

    return {
        "tables_json": str(json_path),
        "tables_csv": str(csv_path),
        "reconstructed_xml": str(xml_path),
        "table_count": str(len(parser.tables)),
    }


def valid_xml_bytes(data: bytes) -> bool:
    try:
        ET.fromstring(data)
        return True
    except ET.ParseError:
        return False


async def get_with_retries(request_context, url: str, headers: dict[str, str]):
    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = await request_context.get(
                url,
                headers=headers,
                timeout=60000,
            )
            if response.status == 200:
                return response
            body = (await response.text())[:300]
            raise RuntimeError(f"HTTP {response.status}: {body}")
        except Exception as exc:
            last_error = exc
            log.warning("Request attempt %d/%d failed: %s", attempt, MAX_RETRIES, exc)
            if attempt < MAX_RETRIES:
                await asyncio.sleep(attempt * 2)
    raise RuntimeError(f"Request failed after retries: {last_error}")


async def download_archive(request_context, url: str, target: Path, expected: str) -> tuple[str, str]:
    if target.exists() and target.stat().st_size > 0:
        return "already_downloaded", str(target)

    response = await get_with_retries(
        request_context,
        url,
        {
            **HEADERS,
            "Accept": "application/xml,text/xml,text/html,*/*",
        },
    )
    body = await response.body()

    if expected == "xml" and not valid_xml_bytes(body):
        raise ValueError("NSE archive returned HTTP 200 but body is not valid XML")
    if expected == "html":
        sample = body[:4096].lower()
        if b"<html" not in sample and b"<!doctype html" not in sample:
            raise ValueError("Legacy detail URL did not return an HTML document")

    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(body)
    return "downloaded", str(target)


def save_manifest(rows: list[dict[str, Any]]) -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "manifest.json").write_text(
        json.dumps(rows, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    fields = sorted({key for row in rows for key in row})
    with (OUT / "manifest.csv").open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


async def fetch_api_json(request_context, from_date: str, to_date: str) -> tuple[Any, str]:
    params = {
        "index": "equities",
        "from_date": from_date,
        "to_date": to_date,
        "symbol": SYMBOL,
        "issuer": ISSUER,
        "period": "Quarterly",
    }
    url = f"{BASE}{API_PATH}?{urlencode(params)}"
    log.info("Calling NSE API: %s", url)

    response = await get_with_retries(request_context, url, HEADERS)
    payload = await response.json()

    OUT.mkdir(parents=True, exist_ok=True)
    raw_name = f"nse_api_{from_date.replace('-', '')}_{to_date.replace('-', '')}.json"
    (OUT / raw_name).write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return payload, url


async def run(headless: bool, from_date: str, to_date: str) -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    manifest: list[dict[str, Any]] = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context(
            locale="en-IN",
            user_agent=HEADERS["User-Agent"],
            viewport={"width": 1440, "height": 1000},
        )
        page = await context.new_page()

        log.info("Opening NSE to establish browser session/cookies...")
        try:
            await page.goto(
                RESULTS_PAGE,
                wait_until="domcontentloaded",
                timeout=60000,
            )
            await page.wait_for_timeout(2500)
        except PlaywrightTimeoutError:
            log.warning("NSE page navigation timed out; trying API with current session.")

        try:
            payload, api_url = await fetch_api_json(context.request, from_date, to_date)
        except Exception:
            # Keep the browser visible if NSE presents a challenge/interstitial.
            if headless:
                await browser.close()
                raise
            log.exception(
                "API request failed. If Chromium shows a challenge, complete it "
                "in the browser, then press ENTER in this terminal."
            )
            await asyncio.to_thread(input, "After NSE page is accessible, press ENTER: ")
            payload, api_url = await fetch_api_json(context.request, from_date, to_date)

        try:
            records = extract_records(payload)
        except Exception:
            await browser.close()
            raise

        selected = filter_records(records)
        log.info(
            "API returned %d records; %d match CANBK standalone quarterly "
            "filings in the requested filing-date window.",
            len(records),
            len(selected),
        )

        if not selected:
            log.warning("No matching records. Inspect the saved raw API JSON.")

        for index, record in enumerate(selected, start=1):
            folder = filing_folder(record)
            stem = filename_stem(record)
            xbrl_url = record.get("xbrl")
            detail_url = record.get("resultDetailedDataLink")

            result: dict[str, Any] = {
                "symbol": SYMBOL,
                "company": record.get("companyName"),
                "isin": record.get("isin") or ISIN,
                "period": record.get("period"),
                "fromDate": record.get("fromDate"),
                "toDate": record.get("toDate"),
                "filingDate": record.get("filingDate"),
                "broadCastDate": record.get("broadCastDate"),
                "financialYear": record.get("financialYear"),
                "relatingTo": record.get("relatingTo"),
                "audited": record.get("audited"),
                "consolidated": record.get("consolidated"),
                "format": record.get("format"),
                "seqNumber": record.get("seqNumber"),
                "params": record.get("params"),
                "identity": filing_identity(record),
                "xbrl_url": xbrl_url,
                "detail_url": detail_url,
                "api_url": api_url,
                "status": "",
                "local_path": "",
                "error": "",
            }

            log.info(
                "[%d/%d] %s | %s | seq=%s | %s",
                index,
                len(selected),
                record.get("toDate"),
                record.get("relatingTo"),
                record.get("seqNumber"),
                record.get("broadCastDate"),
            )

            try:
                if valid_nse_archive_url(xbrl_url, ".xml"):
                    target = folder / f"{stem}.xml"
                    status, path = await download_archive(
                        context.request, xbrl_url, target, "xml"
                    )
                    result["status"] = status
                    result["local_path"] = path
                    result["source_type"] = "ORIGINAL_NSE_XBRL"

                elif valid_nse_archive_url(detail_url, ".html"):
                    html_target = folder / f"{stem}_original.html"
                    status, path = await download_archive(
                        context.request, detail_url, html_target, "html"
                    )
                    result["status"] = status
                    result["local_path"] = path
                    result["source_type"] = "LEGACY_HTML"

                    extracted = save_reconstructed_tables(
                        html_target.read_bytes(), record, folder
                    )
                    result.update(extracted)

                else:
                    result["status"] = "NO_DOWNLOADABLE_SOURCE"
                    result["source_type"] = "UNAVAILABLE"
                    result["error"] = (
                        "Neither a valid NSE XBRL XML URL nor a legacy detail HTML URL "
                        "was supplied by the API."
                    )

            except Exception as exc:
                result["status"] = "FAILED"
                result["error"] = repr(exc)
                log.exception("Failed filing seq=%s", record.get("seqNumber"))

            manifest.append(result)
            save_manifest(manifest)
            await asyncio.sleep(REQUEST_DELAY_SECONDS)

        await browser.close()

    summary = {}
    for row in manifest:
        key = row.get("status", "UNKNOWN")
        summary[key] = summary.get(key, 0) + 1

    log.info("Completed. Summary: %s", summary)
    log.info("Output directory: %s", OUT.resolve())


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Download Canara Bank standalone NSE financial filings."
    )
    parser.add_argument(
        "--from-date",
        default=FROM_DATE,
        help="Filing-date start in DD-MM-YYYY format (default: 01-01-2014)",
    )
    parser.add_argument(
        "--to-date",
        default=TO_DATE,
        help="Filing-date end in DD-MM-YYYY format (default: 08-11-2026)",
    )
    parser.add_argument(
        "--headless",
        action="store_true",
        help="Run Chromium headlessly. Use visible mode first if NSE blocks access.",
    )
    args = parser.parse_args()

    # Validate date strings before launching browser.
    datetime.strptime(args.from_date, "%d-%m-%Y")
    datetime.strptime(args.to_date, "%d-%m-%Y")

    asyncio.run(run(args.headless, args.from_date, args.to_date))



## 3. Configure date range and run
Chromium is visible by default to support NSE session challenges. Files are saved under `canara_bank_xbrl/`. Set `HEADLESS=True` only if NSE allows it.

In [3]:
FROM_DATE = "01-01-2014"
TO_DATE = "08-11-2026"
HEADLESS = False

await run(headless=HEADLESS, from_date=FROM_DATE, to_date=TO_DATE)

2026-11-08 20:17:22,571 | INFO | Opening NSE to establish browser session/cookies...
2026-11-08 20:17:27,178 | INFO | Calling NSE API: https://www.nseindia.com/api/corporates-financial-results?index=equities&from_date=01-01-2014&to_date=08-11-2026&symbol=CANBK&issuer=Canara+Bank&period=Quarterly
2026-11-08 20:17:27,731 | INFO | API returned 66 records; 44 match CANBK standalone quarterly filings in the requested filing-date window.
2026-11-08 20:17:27,734 | INFO | [1/44] 31-Dec-2013 | Third Quarter | seq=114456 | 01-Feb-2014 15:29:02
2026-11-08 20:17:28,827 | INFO | [2/44] 31-Mar-2014 | Fourth Quarter | seq=116757 | 05-May-2014 17:37:02
2026-11-08 20:17:29,600 | INFO | [3/44] 30-Jun-2014 | First Quarter | seq=120714 | 22-Jul-2014 10:17:02
2026-11-08 20:17:30,366 | INFO | [4/44] 30-Sep-2014 | Second Quarter | seq=123560 | 11-Nov-2014 11:25:03
2026-11-08 20:17:31,135 | INFO | [5/44] 31-Dec-2014 | Third Quarter | seq=125640 | 04-Feb-2015 17:37:02
2026-11-08 20:17:31,886 | INFO | [6/44] 31

## 4. Inspect manifest and download status

In [4]:
import pandas as pd
manifest_path = OUT / "manifest.csv"
if manifest_path.exists():
    df = pd.read_csv(manifest_path)
    display(df[[c for c in ["toDate", "broadCastDate", "consolidated", "source_type", "status", "local_path", "error"] if c in df.columns]])
    print("\nStatus counts:")
    display(df["status"].value_counts(dropna=False))
else:
    print("No manifest yet. Run cell 3 first.")

,toDate,broadCastDate,consolidated,source_type,status,local_path,error
0,31-Dec-2013,01-Feb-2014 15:29:02,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2014/CANBK_20131231_Third_Qua...,NaN
1,31-Mar-2014,05-May-2014 17:37:02,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2014/CANBK_20140331_Fourth_Qu...,NaN
2,30-Jun-2014,22-Jul-2014 10:17:02,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2014/CANBK_20140630_First_Qua...,NaN
3,30-Sep-2014,11-Nov-2014 11:25:03,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2014/CANBK_20140930_Second_Qu...,NaN
4,31-Dec-2014,04-Feb-2015 17:37:02,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2015/CANBK_20141231_Third_Qua...,NaN
5,31-Mar-2015,25-May-2015 17:05:02,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2015/CANBK_20150331_Fourth_Qu...,NaN
6,30-Jun-2015,05-Aug-2015 16:20:14,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2015/CANBK_20150630_First_Qua...,NaN
7,30-Sep-2015,05-Nov-2015 11:34:43,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2015/CANBK_20150930_Second_Qu...,NaN
8,31-Dec-2015,13-Feb-2016 18:41:19,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2016/CANBK_20151231_Third_Qua...,NaN
9,31-Mar-2016,29-May-2016 16:54:16,Non-Consolidated,LEGACY_HTML,downloaded,canara_bank_xbrl/2016/CANBK_20160331_Fourth_Qu...,NaN



Status counts:


status
downloaded                39
FAILED                     4
NO_DOWNLOADABLE_SOURCE     1
Name: count, dtype: int64

## Notes
- The API date filter is treated as a **filing-date** window; older fiscal periods may be filed later.
- `xbrl` URLs are downloaded as original XML when provided. HTML tables are **not equivalent to XBRL taxonomy facts**.
- The notebook has not been executed against NSE here; actual availability depends on the NSE response and session.
- Do not paste browser session cookies into this notebook.